# Sentinel-1 and Sentinel-2 parcel zonal statistics

Configure and run `OpenEOJobManagerZonalStats` through the multi-user scheduler. openEO creates sensor-preprocessed physical-band NetCDF cubes per spatial batch. IQR cleaning, null filling, pixel indices, and parcel statistics run locally, producing one GeoParquet row per parcel with dated features and a separate annual summary. See [the detailed workflow](../../../docs/data_preparation/openeo/WORKFLOW.md).

In [1]:
from pathlib import Path
import os
import shutil
from IPython.display import display
import geopandas as gpd
from datetime import datetime
from data_preparation.parcel_stats import estimate_utm_epsg_from_parcels
from data_preparation.parcel_stats.job_manager import compute_tile_buffer_metres
from data_preparation.parcel_stats.multiuser import (
    load_openeo_users_from_db,
)
from shared.io import read_data, write_data, file_exists
from shared.spatial_statistics import spatial_clustering_using_buffer, spatial_clustering_using_grid
import shared.geometry as geom_l

from data_preparation.features import (
    append_parcel_elevation,
    append_soil_feature,
    create_dem_mosaic,
    find_dem_tiles,
)

In [2]:
class StopExecution(Exception):
    """Stops the notebook run silently, without a traceback."""
    def _render_traceback_(self):
        return []

## Inputs

Processing order:

`Sentinel-2 L2A`
`acquisition cloud mask`
`optional temporal reduction, or retain acquisitions`
`per-band, per-time-slice raster IQR outlier removal when enabled`
`local null filling`
`local pixel indices from cleaned physical bands`
`parcel statistics`
`one-row-per-parcel time-series pivot`
`ML-ready GeoParquet`

Temporal filling is past-only so later observations cannot leak into earlier feature periods.

In [3]:
# ============================================================================
# STEP 1: Read data and re-project
# ============================================================================
# Read the data and re-project to the desired EPSG

WORK_PATH = Path("C:/work_sat")
RUN_IDENTIFIER = "central_greece"

PARCELS_PATH = WORK_PATH / RUN_IDENTIFIER / "central_greece_prediction.parquet"
gdf_parcels = read_data(str(PARCELS_PATH))
print("Initial parcels data:")
display(gdf_parcels.head())
display(gdf_parcels.columns)

keep_columns = ["parcel_id", "crop_code", "crop_label", "pred_crop_code", "pred_crop_label", "pred_probability", "set_type",
                "svm_is_outlier", "geometry"]
gdf_parcels = gdf_parcels[keep_columns].copy()
display(gdf_parcels.head())
print(f"Parcels before dropping duplicate geometries: {len(gdf_parcels)}")
gdf_parcels = gdf_parcels.drop_duplicates(subset=["geometry"]).copy()
print(f"Parcels after dropping duplicate geometries: {len(gdf_parcels)}")

gdf_parcels = geom_l.return_valid_geometries(gdf_parcels)
gdf_parcels = geom_l.convert_multipolygons_to_polygons(gdf_parcels)
print(f"Number of parcels after ensuring valid geometries and converting multipolygons to polygons: {len(gdf_parcels)}")

display(gdf_parcels)

Reading file: C:/work_sat/central_greece/central_greece_prediction.parquet
Initial parcels data:


,parcel_id,parcel_multiyear_id,year,crop_code,crop_label,pred_crop_code,pred_crop_label,pred_probability,pred_correct,set_type,...,cocultivation,sowing,zone_id,s2_tile_id,cartographic_basemap,area_ha,shape_area,shape_length,pix_count,geometry
0,753,10251480,2024,108,nut_trees,108,nut_trees,0.441608,True,train,...,False,False,2,24,3963979322004,0.961193,9611.928421,529.575797,32.0,"MULTIPOLYGON (((396871.63 4397265.407, 396875...."
1,136423,7588665,2024,101,olive_groves,101,olive_groves,0.723092,True,test,...,False,False,2,24,3764049003002,0.401124,4011.240489,276.755568,9.0,"MULTIPOLYGON (((376861.31 4403934.534, 376851...."
2,204783,6231558,2024,101,olive_groves,101,olive_groves,0.710824,True,test,...,False,False,2,24,3774000994007,0.467767,4677.673439,404.692436,8.0,"MULTIPOLYGON (((377236.773 4401113.632, 377254..."
3,245497,10770682,2024,101,olive_groves,129,fallow,0.545589,False,test,...,False,False,2,24,3814232856003,0.353199,3531.994102,238.988690,9.0,"MULTIPOLYGON (((381314.955 4423763.509, 381299..."
4,797263,9353904,2024,102,grape_vines,102,grape_vines,0.750356,True,test,...,False,False,2,24,3603984176030,0.383018,3830.181429,263.445246,9.0,"MULTIPOLYGON (((360493.572 4398483.286, 360554..."


Index(['parcel_id', 'parcel_multiyear_id', 'year', 'crop_code', 'crop_label',
       'pred_crop_code', 'pred_crop_label', 'pred_probability', 'pred_correct',
       'set_type', 'in_holdout', 'holdout_full', 'svm_is_outlier',
       'svm_outlier_full', 'lpis_normal', 'has_rfv', 'rfv_true',
       'declared_crop_type_id', 'declared_crop_variation_id', 'cocultivation',
       'sowing', 'zone_id', 's2_tile_id', 'cartographic_basemap', 'area_ha',
       'shape_area', 'shape_length', 'pix_count', 'geometry'],
      dtype='object')

,parcel_id,crop_code,crop_label,pred_crop_code,pred_crop_label,pred_probability,set_type,svm_is_outlier,geometry
0,753,108,nut_trees,108,nut_trees,0.441608,train,False,"MULTIPOLYGON (((396871.63 4397265.407, 396875...."
1,136423,101,olive_groves,101,olive_groves,0.723092,test,False,"MULTIPOLYGON (((376861.31 4403934.534, 376851...."
2,204783,101,olive_groves,101,olive_groves,0.710824,test,False,"MULTIPOLYGON (((377236.773 4401113.632, 377254..."
3,245497,101,olive_groves,129,fallow,0.545589,test,False,"MULTIPOLYGON (((381314.955 4423763.509, 381299..."
4,797263,102,grape_vines,102,grape_vines,0.750356,test,False,"MULTIPOLYGON (((360493.572 4398483.286, 360554..."


Parcels before dropping duplicate geometries: 23073
Parcels after dropping duplicate geometries: 23073
Number of parcels after ensuring valid geometries and converting multipolygons to polygons: 23073


,parcel_id,crop_code,crop_label,pred_crop_code,pred_crop_label,pred_probability,set_type,svm_is_outlier,geometry
0,753,108,nut_trees,108,nut_trees,0.441608,train,False,"POLYGON ((396871.63 4397265.407, 396875.582 43..."
1,136423,101,olive_groves,101,olive_groves,0.723092,test,False,"POLYGON ((376861.31 4403934.534, 376851.653 44..."
2,204783,101,olive_groves,101,olive_groves,0.710824,test,False,"POLYGON ((377236.773 4401113.632, 377254.798 4..."
3,245497,101,olive_groves,129,fallow,0.545589,test,False,"POLYGON ((381314.955 4423763.509, 381299.158 4..."
4,797263,102,grape_vines,102,grape_vines,0.750356,test,False,"POLYGON ((360493.572 4398483.286, 360554.752 4..."
...,...,...,...,...,...,...,...,...,...
23068,294878979,130,pastures,130,pastures,0.564301,test,False,"POLYGON ((366748.995 4402517.412, 366748.988 4..."
23069,294878980,130,pastures,130,pastures,0.663258,train,False,"POLYGON ((366547.416 4403202.695, 366709.21 44..."
23070,294878981,101,olive_groves,130,pastures,0.341065,test,True,"POLYGON ((366561.698 4402366.616, 366577.042 4..."
23071,294878983,130,pastures,130,pastures,0.650957,train,False,"POLYGON ((367460.882 4403400.962, 367468.411 4..."


In [4]:
print(gdf_parcels.crs)
WORKING_EPSG = gdf_parcels.crs.to_epsg()
# WORKING_EPSG = estimate_utm_epsg_from_parcels(gdf_parcels)
# WORKING_EPSG

{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "ProjectedCRS", "name": "GGRS87 / Greek Grid", "base_crs": {"name": "GGRS87", "datum": {"type": "GeodeticReferenceFrame", "name": "Greek Geodetic Reference System 1987", "ellipsoid": {"name": "GRS 1980", "semi_major_axis": 6378137, "inverse_flattening": 298.257222101}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "id": {"authority": "EPSG", "code": 4121}}, "conversion": {"name": "Greek Grid", "method": {"name": "Transverse Mercator", "id": {"authority": "EPSG", "code": 9807}}, "parameters": [{"name": "Latitude of natural origin", "value": 0, "unit": "degree", "id": {"authority": "EPSG", "code": 8801}}, {"name": "Longitude of natural origin", "value": 24, "unit": "degree", "id": {"authority": "EPSG", "code": 8802}}

In [5]:
# ============================================================================
# Define the class column and parcel ID column names.
# These are used in the extraction process to identify the class of each parcel
# and its unique identifier.
# ============================================================================
class_column = "crop_label"
parcel_id_column = "parcel_id"

In [6]:
label_counts = gdf_parcels[class_column].value_counts()
rare_labels = label_counts[label_counts < 10].index
gdf_parcels.loc[gdf_parcels[class_column].isin(rare_labels), class_column] = "9999"
print(f"Number of unique labels after grouping rare labels: {len(gdf_parcels[class_column].value_counts())}")
print(gdf_parcels[class_column].value_counts())

cond = gdf_parcels[class_column] == "9999"
print(50*"-")
print(f"Excluded parcels with label '9999': {cond.sum()} out of {len(gdf_parcels)}")
gdf_parcels = gdf_parcels[~cond].copy()
print(f"Selected parcels after excluding label '9999': {len(gdf_parcels)}")

# print(50*"-")
# exclude_categories = ["tobacco", "aromatic", "grass_fodders", "horticultural_fall"]
# cond = gdf_parcels[class_column].isin(exclude_categories)
# print(f"Excluded parcels with label {exclude_categories}: {cond.sum()} out of {len(gdf_parcels)}")
# gdf_parcels = gdf_parcels[~cond].copy()
# print(f"Selected parcels after excluding label {exclude_categories}: {len(gdf_parcels)}")
# print(gdf_parcels[class_column].value_counts())

current_datetime = datetime.now().strftime("%Y%m%d_%H%M%S")
PARCELS_PATH = WORK_PATH / RUN_IDENTIFIER / f"cleaned_grouped_parcels_{current_datetime}.gpkg"
write_data(gdf_parcels, str(PARCELS_PATH))

Number of unique labels after grouping rare labels: 15
crop_label
olive_groves         5162
nut_trees            4477
wheat                3275
fallow               2222
grape_vines          1832
stone_fruit_trees    1812
pome_trees           1404
pastures              892
kiwi                  852
cotton                569
other_trees           400
tobacco                79
aromatic               50
grass_fodders          46
9999                    1
Name: count, dtype: int64
--------------------------------------------------
Excluded parcels with label '9999': 1 out of 23073
Selected parcels after excluding label '9999': 23072
Writing file: C:/work_sat/central_greece/cleaned_grouped_parcels_20261124_094211.gpkg


In [7]:
train_df = gdf_parcels[gdf_parcels["set_type"] == "train"].copy()
test_df = gdf_parcels[gdf_parcels["set_type"] == "test"].copy()

train_accuracy = (train_df["crop_code"] == train_df["pred_crop_code"]).mean()
print(f"Train accuracy: {100*train_accuracy:.4f}")
test_accuracy = (test_df["crop_code"] == test_df["pred_crop_code"]).mean()
print(f"Test accuracy: {100*test_accuracy:.4f}")

# total_accuracy = (gdf_parcels["crop_code"] == gdf_parcels["pred_crop_code"]).mean()
# print(f"Total accuracy: {total_accuracy:.4f}")

Train accuracy: 85.2633
Test accuracy: 68.6386


# Add centroid

In [8]:
# This can be added in the web application
gdf_parcels["centroid_x"] = gdf_parcels.centroid.x
gdf_parcels["centroid_y"] = gdf_parcels.centroid.y
write_data(gdf_parcels, str(PARCELS_PATH))

Removing existing file: C:/work_sat/central_greece/cleaned_grouped_parcels_20261124_094211.gpkg
Writing file: C:/work_sat/central_greece/cleaned_grouped_parcels_20261124_094211.gpkg


# Add Soil

In [9]:
SOIL_PATH = Path("F:/parcels/Agro_Soil_Clean.gpkg")
SOIL_LAYER = "agrosoil"

if file_exists(str(SOIL_PATH)):
    gdf_parcels = append_soil_feature(
        parcels_path=PARCELS_PATH,
        soil_path=SOIL_PATH,
        output_path=PARCELS_PATH,
        soil_layer=SOIL_LAYER,
        attribute_column="GAIOIKANOT",
        overwrite=True,
    )
    assert gdf_parcels.crs == WORKING_EPSG

    write_data(gdf_parcels, str(PARCELS_PATH))

File 'F:\parcels\Agro_Soil_Clean.gpkg' does not exist.


# Add Elevation

In [10]:
DEM_DIR = WORK_PATH / RUN_IDENTIFIER / "dem"
DEM_DIR.mkdir(parents=True, exist_ok=True)
MOSAIC_PATH = DEM_DIR / "dem_mosaic.tif"

EPSG_CODE = 2100  # Greek Grid (metric)
# False uses pixels whose centers fall inside each parcel. Set True for very small parcels.
ALL_TOUCHED = False

In [12]:
# def_files_path = "F:/esahub/Processed/data_common/dem_fl/ESA_10"
# gdf_dem_tiles_path = os.path.join(def_files_path, "esa_tiles_dem_10m.gpkg")
# gdf_dem_tiles = read_data(gdf_dem_tiles_path)
# gdf_dem_tiles = gdf_dem_tiles.to_crs(epsg=EPSG_CODE)
# tiles_to_copy = gdf_dem_tiles[gdf_dem_tiles.geometry.intersects(gdf_parcels.union_all())]
# display(tiles_to_copy)
# for _, tile in tiles_to_copy.iterrows():
#     print(f"Processing tile: {tile}")
#     src_tile_path = os.path.join(def_files_path, tile.file_name)
#     dst_tile_path = os.path.join(DEM_DIR, tile.file_name)
#     if not os.path.isfile(dst_tile_path):
#         print(f"Copying {src_tile_path} to {dst_tile_path}")
#         shutil.copy(src_tile_path, dst_tile_path)

# dem_tiles = find_dem_tiles(DEM_DIR, MOSAIC_PATH)
# # Reuses a valid existing mosaic. Set overwrite=True to rebuild it.
# MOSAIC_PATH = create_dem_mosaic(dem_tiles, MOSAIC_PATH, overwrite=False)

In [11]:
if file_exists(str(MOSAIC_PATH)):
    parcels_with_elevation = append_parcel_elevation(
        parcels_path=PARCELS_PATH,
        mosaic_path=MOSAIC_PATH,
        output_path=PARCELS_PATH,
        all_touched=ALL_TOUCHED,
        overwrite=True,
    )
    assert parcels_with_elevation.crs == WORKING_EPSG
    gdf_parcels = parcels_with_elevation.copy()
    write_data(gdf_parcels, str(PARCELS_PATH))

Reading file: C:/work_sat/central_greece/cleaned_grouped_parcels_20261124_094211.gpkg
Parcel dataset CRS: EPSG:2100
Parcel CRS type: <class 'pyproj.crs.crs.CRS'>
Parcel CRS to_epsg(): 2100
Parcel CRS is_projected: True
Reprojecting parcel geometries from EPSG:2100 to EPSG:4326 for zonal statistics.
Processed 5,000/23,072 parcels
Processed 10,000/23,072 parcels
Processed 15,000/23,072 parcels
Processed 20,000/23,072 parcels
Processed 23,072/23,072 parcels
Writing file: C:/work_sat/central_greece/.cleaned_grouped_parcels_20261124_094211.tmp.gpkg
Saved parcels with mean elevation: C:\work_sat\central_greece\cleaned_grouped_parcels_20261124_094211.gpkg
Parcels without a valid DEM pixel: 0/23,072
Removing existing file: C:/work_sat/central_greece/cleaned_grouped_parcels_20261124_094211.gpkg
Writing file: C:/work_sat/central_greece/cleaned_grouped_parcels_20261124_094211.gpkg


# Add cluster, only if the number of parcels is greater than 2, otherwise skip this step

In [14]:
# # Reuse saved cluster labels; generate them only when the input has none.
# CLUSTER_FIELD = "cluster_buffer"
# if CLUSTER_FIELD not in gdf_parcels.columns:
#     gdf_grouped = spatial_clustering_using_buffer(
#         gdf_parcels.to_crs(epsg=WORKING_EPSG),
#         parcel_id_column,
#         min_cluster_distance_in_m=5_000,
#         field_cluster=CLUSTER_FIELD,
#         plot_results=False,
#     )
#     if gdf_grouped[CLUSTER_FIELD].isna().any():
#         print("Every parcel must have a cluster label before partitioning.")
#         raise StopExecution
#     if len(gdf_grouped) != len(gdf_parcels):
#         print("The number of parcels after clustering does not match the original count.")
#         raise StopExecution
#     if len(gdf_grouped[CLUSTER_FIELD].unique()) < 2:
#         print("Clustering resulted in fewer than 2 unique clusters, which is insufficient for partitioning.")
#         raise StopExecution
#     gdf_parcels = gdf_grouped.copy()
#     write_data(gdf_parcels, str(PARCELS_PATH))

In [12]:
# Split the parcels based on a grid.
# Each parcel is labelled by the grid cell that contains its centroid; only non-empty
# cells get a label (0..n-1). This overwrites any earlier CLUSTER_FIELD labels.
CLUSTER_FIELD = "cluster_grid"
if CLUSTER_FIELD not in gdf_parcels.columns:
    cols = 10
    rows = 10

    gdf_grouped = spatial_clustering_using_grid(
        gdf_parcels,
        parcel_id_column,
        rows=rows,
        cols=cols,
        field_cluster=CLUSTER_FIELD,
        plot_results=False,
    )
    print(f"Assigned {len(gdf_grouped)} parcels to {gdf_grouped[CLUSTER_FIELD].nunique()} grid parts")

    # Later cells partition gdf_grouped by CLUSTER_FIELD.
    gdf_parcels = gdf_grouped.copy()
    write_data(gdf_parcels, str(PARCELS_PATH))

Spatial grid: 10x10 cells of 6205 x 6112 data units


,cluster_grid,parcel_id,total
0,0,"[24841285, 28095143, 42152750, 129405200, 1300...",142
1,1,"[6883111, 50836716, 50836723, 50836733, 508367...",266
2,2,"[2345699, 2956552, 3621588, 8129981, 13960556,...",435
3,3,"[6155113, 6379749, 6841465, 16261124, 23761243...",206
4,4,"[42121853, 49790588, 51876574, 95785790, 10673...",247
...,...,...,...
64,64,"[30059251, 41888550, 48274712, 167140887, 1752...",12
65,65,"[256835333, 256835334, 256835335]",3
66,66,"[2240774, 46479833, 46479854, 49844880, 518903...",55
67,67,"[35049231, 36820299, 143042930, 235395192, 266...",8


Assigned 23072 parcels to 69 grid parts
Removing existing file: C:/work_sat/central_greece/cleaned_grouped_parcels_20261124_094211.gpkg
Writing file: C:/work_sat/central_greece/cleaned_grouped_parcels_20261124_094211.gpkg


## Multi-user extraction planning

Accounts are loaded from PostgreSQL using `load_openeo_users_from_db(app_user_id=APP_USER_ID)`. Set the owner ID
in the next cell or provide `APP_USER_ID` in the kernel environment. Install the `database` and `credentials` extras.
Provide `DB_NAME` and `POSTGRES_*` in the kernel environment. The account cell reads the web app's
`SATELLITES_CREDENTIAL_KEY` from `.env` in the working directory or its parents, without changing database settings.
Install `python-dotenv` in the notebook kernel if needed (`%pip install python-dotenv`); never print the key.
`TBL_APP_USERS` and `TBL_EO_USERS` default to `public.app_users` and `public.eo_users`. `TBL_USERS` is ignored.
Only the active owner's accounts are read; passwords are decrypted in memory. Do not print or save credentials.
Accounts are sorted by username. Explicitly restore the original account order when resuming legacy jobs
without saved owners or using partition scheduling. See [database account setup](../../../docs/data_preparation/openeo/CREDENTIALS.md).

Create one spatial partition per cluster, then plan all tile batches before
submission. A shared queue lets every account process batches from any partition. With `fit_tiles_to_parcels=True`, a cube covers the
group's complete bounding box plus `TILE_BUFFER_METRES` on every side:

- width = east - west + 2 * buffer
- height = north - south + 2 * buffer

Only if either dimension exceeds `TILE_SIZE_METRES` does the planner split the
group spatially, then fit and check each subgroup in the same way. Parcels stay
whole and belong to exactly one job. A single parcel that exceeds the limit with
its buffer raises an error before submission.

A large partition can use all available accounts at once. Each account stays
within OPENEO_JOBS_PER_USER. Existing cubes are reused; previously submitted
jobs resume on their owning account. Keep the original account order when
first resuming a legacy run, which did not record account names.

Partition outputs and the merge step stay the same. Local statistics run after
the shared download queue completes, using LOCAL_BATCH_WORKERS per active
partition (up to one active partition per account).


In [13]:
# ============================================================================
# STEP 2: Load accounts for the shared batch queue
# ============================================================================
# Accounts share unsubmitted batches from all cluster partitions.

from dotenv import dotenv_values, find_dotenv

# Refresh only the encryption key; keep the notebook's existing database connection settings.
credential_env_file = find_dotenv(usecwd=True)
if credential_env_file:
    credential_key = dotenv_values(credential_env_file).get("SATELLITES_CREDENTIAL_KEY")
    if credential_key is not None:
        os.environ["SATELLITES_CREDENTIAL_KEY"] = credential_key
    del credential_key

OPENEO_JOBS_PER_USER = 2  # This is default by openEO
APP_USER_ID = 3  # Selected app_users owner; use None to read APP_USER_ID from the kernel environment.
if APP_USER_ID is None:
    owner_setting = os.getenv("APP_USER_ID", "").strip()
    if not owner_setting.isdecimal() or int(owner_setting) <= 0:
        raise ValueError("Error: Set APP_USER_ID to your positive app_users.app_user_id before loading accounts.")
    APP_USER_ID = int(owner_setting)
users_list = load_openeo_users_from_db(app_user_id=APP_USER_ID)
print(f"Loaded {len(users_list)} openEO user(s):")
for i, (user, _) in enumerate(users_list, 1):
    print(f"  User {i}: {user}")

Loaded 14 openEO user(s) for the application user.
Loaded 14 openEO user(s):
  User 1: esagovhub1@gmail.com
  User 2: esagovhub2@gmail.com
  User 3: esagovhub3@gmail.com
  User 4: esagovhub4@gmail.com
  User 5: farfarasmpoytos@gmail.com
  User 6: grpkhub_a@outlook.com
  User 7: grpkhubb@outlook.com
  User 8: grpkhubc@outlook.com
  User 9: grpkhubd@outlook.com
  User 10: grpkhube@outlook.com
  User 11: grpkhubf@outlook.com
  User 12: katrinporo@gmail.com
  User 13: louvarieleni@gmail.com
  User 14: sofiaargirou03@gmail.com


In [14]:
if CLUSTER_FIELD in gdf_grouped.columns:
    # Keep each cluster together; the existing tile planner handles its size limit.
    spatial_parts = [
        part.copy()
        for _, part in gdf_grouped.groupby(CLUSTER_FIELD, sort=True, observed=True)
    ]
    assert sum(len(part) for part in spatial_parts) == len(gdf_grouped)

    print(f"Created {len(spatial_parts)} cluster partitions:")
    for i, part in enumerate(spatial_parts, 1):
        print(f"  Partition {i}: {part[CLUSTER_FIELD].iloc[0]} - {len(part)} parcels")


Created 69 cluster partitions:
  Partition 1: 0 - 142 parcels
  Partition 2: 1 - 266 parcels
  Partition 3: 2 - 435 parcels
  Partition 4: 3 - 206 parcels
  Partition 5: 4 - 247 parcels
  Partition 6: 5 - 323 parcels
  Partition 7: 6 - 183 parcels
  Partition 8: 7 - 544 parcels
  Partition 9: 8 - 463 parcels
  Partition 10: 9 - 136 parcels
  Partition 11: 10 - 246 parcels
  Partition 12: 11 - 659 parcels
  Partition 13: 12 - 902 parcels
  Partition 14: 13 - 385 parcels
  Partition 15: 14 - 433 parcels
  Partition 16: 15 - 735 parcels
  Partition 17: 16 - 85 parcels
  Partition 18: 17 - 243 parcels
  Partition 19: 18 - 519 parcels
  Partition 20: 19 - 497 parcels
  Partition 21: 20 - 74 parcels
  Partition 22: 21 - 816 parcels
  Partition 23: 22 - 1475 parcels
  Partition 24: 23 - 550 parcels
  Partition 25: 24 - 826 parcels
  Partition 26: 25 - 1326 parcels
  Partition 27: 26 - 28 parcels
  Partition 28: 27 - 26 parcels
  Partition 29: 28 - 338 parcels
  Partition 30: 29 - 638 parcels


In [15]:
# Maximum requested cube width AND height, including the buffer.
# Each group gets its own fitted extent; only oversized groups are split.
TILE_SIZE_METRES = 5_400
TILE_BUFFER_METRES = compute_tile_buffer_metres(parcels=gdf_grouped, working_epsg=WORKING_EPSG, safety_factor=1)

print(f"Maximum buffered cube width/height: {TILE_SIZE_METRES:,} m")
print(f"Buffer on each side: {TILE_BUFFER_METRES:,} m")


Recommended tile buffer: 1,700 m
Maximum buffered cube width/height: 5,400 m
Buffer on each side: 1,700 m


In [16]:
# ============================================================================
# Confirm the planned global remote concurrency.
# ============================================================================
TOTAL_REMOTE_JOB_CAPACITY = len(users_list) * OPENEO_JOBS_PER_USER
print(
    f"Remote capacity: {len(users_list)} users x {OPENEO_JOBS_PER_USER} jobs "
    f"= {TOTAL_REMOTE_JOB_CAPACITY} simultaneous jobs"
)


Remote capacity: 14 users x 2 jobs = 28 simultaneous jobs


## How this grid split becomes openEO jobs

The next two cells recompute the figures from the inputs above: `spatial_parts` (from `CLUSTER_FIELD`),
`TILE_SIZE_METRES`, `TILE_BUFFER_METRES`, `users_list`, `OPENEO_JOBS_PER_USER` and `BATCH_WORKERS`.
Rerun them after changing any of these. Nothing is submitted and no output folders are created: the planner
is the extractor's own fitted tile planner, called directly.

The figures assume the extraction settings in [1_parcel_stats_openEO_fill.ipynb](1_parcel_stats_openEO_fill.ipynb):
`fit_tiles_to_parcels=True`, `scheduling="batches"` and `remove_nc_after_completion=True`. Keep `TILE_SIZE_METRES`
and `BATCH_WORKERS` equal to that notebook's values. The figures also assume a fresh run: on resume, completed
partitions and cached batches are skipped, and submitted jobs stay on their owning account.
See [scheduling](../../../docs/data_preparation/openeo/SCHEDULING.md) and
[parcel grouping](../../../docs/data_preparation/openeo/PARCEL_GROUPING.md).

**Partitions → batches.** Each partition's cube covers its parcels' bounding box plus `TILE_BUFFER_METRES` on every
side. Only cubes wider or taller than `TILE_SIZE_METRES` are split, at the midpoint of the longer side. Parcels always
stay whole, so every parcel's width and height plus `2 * TILE_BUFFER_METRES` must fit in `TILE_SIZE_METRES`;
otherwise planning stops, lists the parcels that do not fit and prints the minimum tile size.
Cube area, not parcel count, drives openEO processing and download size.

**Batches → accounts (`scheduling="batches"`).** All batches are planned first and placed in one FIFO queue, in
partition order and then tile order. One slot thread per account job (`OPENEO_JOBS_PER_USER` per account) takes the
next batch whenever it is free. The rounds below assume equal batch durations. They are not synchronised waves:
faster slots process more batches, and batches of a split partition can run on different accounts. Each submitted
batch records its account in `openeo_user` in its partition's `tile_jobs.parquet`.

**Local processing.** Each downloaded cube goes straight to one run-wide pool of `BATCH_WORKERS` local processes.
These clean the cube, fill gaps, compute parcel statistics, write a checkpoint and delete the NetCDFs. When all
processes are busy, the slot that just finished downloading waits before taking its next batch. Partition GeoParquet
outputs are assembled under `partition_<n>/` after all remote and local work succeeds.

In [17]:
# ============================================================================
# Plan the openEO batches exactly as the extraction will (nothing is submitted)
# ============================================================================
import logging
import math
from types import SimpleNamespace
import pandas as pd
from IPython.display import Markdown
from data_preparation.parcel_stats.core.parcel_batches import ParcelBatchPlanner
from data_preparation.parcel_stats.job_manager import OpenEOJobManagerZonalStats

BATCH_WORKERS = 2  # Match extraction_config["batch_workers"] in 1_parcel_stats_openEO_fill.ipynb

# A cube is a parcel group's bounds plus the buffer on BOTH sides, so even a single parcel
# needs its width/height + 2 * TILE_BUFFER_METRES to fit inside TILE_SIZE_METRES.
parcel_bounds = gdf_grouped.to_crs(epsg=WORKING_EPSG).geometry.bounds
parcel_size = pd.DataFrame({
    parcel_id_column: gdf_grouped[parcel_id_column].to_numpy(),
    "width_m": (parcel_bounds.maxx - parcel_bounds.minx).to_numpy(),
    "height_m": (parcel_bounds.maxy - parcel_bounds.miny).to_numpy(),
})
parcel_size["required_tile_m"] = parcel_size[["width_m", "height_m"]].max(axis=1) + 2 * TILE_BUFFER_METRES
minimum_tile_metres = int(math.ceil(parcel_size["required_tile_m"].max() / 100) * 100)
too_large = parcel_size.loc[parcel_size["required_tile_m"] > TILE_SIZE_METRES]
if 2 * TILE_BUFFER_METRES >= TILE_SIZE_METRES or not too_large.empty:
    print(
        f"TILE_SIZE_METRES = {TILE_SIZE_METRES:,} m cannot hold a {TILE_BUFFER_METRES:,} m buffer on each side "
        f"({2 * TILE_BUFFER_METRES:,} m) plus the largest parcel. Minimum TILE_SIZE_METRES: {minimum_tile_metres:,} m.\n"
        f"{len(too_large)} parcel(s) do not fit; increase TILE_SIZE_METRES, lower TILE_BUFFER_METRES "
        "or handle these parcels separately:"
    )
    display(too_large.sort_values("required_tile_m", ascending=False).round(0).head(20))
    raise StopExecution
print(f"Largest parcel plus buffer needs {minimum_tile_metres:,} m; TILE_SIZE_METRES = {TILE_SIZE_METRES:,} m.")

# Prepare parcels as the extractor does, then call its fitted planner without creating folders or logs.
preparer = ParcelBatchPlanner()
preparer.PARCEL_ID_FIELD = parcel_id_column
preparer.logger = logging.getLogger(__name__)
batch_rows = []
for part_idx, part in enumerate(spatial_parts, start=1):
    planner = SimpleNamespace(
        parcels=preparer._prepare_parcels(part),
        working_epsg=WORKING_EPSG,
        PARCEL_ID_FIELD=parcel_id_column,
        tile_size_metres=TILE_SIZE_METRES,
        tile_buffer_metres=TILE_BUFFER_METRES,
    )
    for plan in OpenEOJobManagerZonalStats._build_fitted_tile_plan(planner, Path(".")):
        extent = plan["spatial_extent"]
        batch_rows.append({
            "partition": part_idx,
            "label": part[CLUSTER_FIELD].iloc[0],
            "batch": plan["batch_number"],
            "parcels": len(plan["parcels"]),
            "width_km": (extent["east"] - extent["west"]) / 1_000,
            "height_km": (extent["north"] - extent["south"]) / 1_000,
        })

# Queue order of a fresh run: partition order, then tile order.
batch_plan = pd.DataFrame(batch_rows)
batch_plan["area_km2"] = batch_plan["width_km"] * batch_plan["height_km"]
batch_plan.index = pd.RangeIndex(1, len(batch_plan) + 1, name="queue_position")
print(f"Planned {len(batch_plan)} batches from {len(spatial_parts)} partitions.")

Largest parcel plus buffer needs 5,400 m; TILE_SIZE_METRES = 5,400 m.
Planned 798 batches from 69 partitions.


In [19]:
# ============================================================================
# Live summary: partitions -> batches -> accounts -> local processing
# ============================================================================
def format_partitions(numbers, limit=12):
    """Return a short comma-separated list of partition numbers."""
    shown = ", ".join(str(number) for number in numbers[:limit])
    return shown if len(numbers) <= limit else f"{shown}, ... (+{len(numbers) - limit})"

total_batches = len(batch_plan)
account_count = len(users_list)
slot_count = account_count * OPENEO_JOBS_PER_USER
batches_per_partition = batch_plan.groupby("partition").size()

# 1. Partitions -> batches
split_rows = []
for count, members in batches_per_partition.groupby(batches_per_partition):
    numbers = members.index.tolist()
    names = f"{len(numbers)} partitions" if count == 1 and len(numbers) > 1 else format_partitions(numbers)
    split_rows.append(f"| {names} | {count} | {count * len(numbers):,} |")
sides = pd.concat([batch_plan["width_km"], batch_plan["height_km"]])
largest = batch_plan.loc[batch_plan["parcels"].idxmax(), ["parcels", "partition"]].astype(int)
west, south, east, north = gdf_grouped.to_crs(epsg=WORKING_EPSG).total_bounds
extent_km2 = (east - west) * (north - south) / 1e6
requested_km2 = batch_plan["area_km2"].sum()

# 2. Batches -> accounts: equal-duration rounds of the FIFO queue
def span(first, last):
    """Return 'first–last', or one value when both are equal."""
    return f"{first}" if first == last else f"{first}–{last}"

round_count = -(-total_batches // slot_count)
# Show the first two rounds and the last one; collapse any full rounds between them.
shown_rounds = [(r, r) for r in range(min(round_count, 2))]
if round_count > 3:
    shown_rounds.append((2, round_count - 2))
if round_count > 2:
    shown_rounds.append((round_count - 1, round_count - 1))
round_rows = []
for first, last in shown_rounds:
    batches = batch_plan.iloc[first * slot_count:(last + 1) * slot_count]
    used = len(batches) - (last - first) * slot_count
    label = "immediate" if first == 0 else ("tail" if used < slot_count else "as slots free")
    idle = f"; {slot_count - used} idle" if used < slot_count else f" ({OPENEO_JOBS_PER_USER} per account)"
    each = " each" if last > first else ""
    round_rows.append(
        f"| {span(first + 1, last + 1)} ({label}) | batches {span(batches.index[0], batches.index[-1])} "
        f"| {span(batches['partition'].iloc[0], batches['partition'].iloc[-1])} | {used}{idle}{each} |"
    )

# 3. Local processing
if BATCH_WORKERS < min(slot_count, total_batches):
    local_note = (
        f"Up to {min(slot_count, total_batches)} cubes can download concurrently but only {BATCH_WORKERS} are processed "
        "locally at once, so after the first round local throughput, not openEO capacity, may set the pace."
    )
else:
    local_note = (
        f"{BATCH_WORKERS} local processes cover all {min(slot_count, total_batches)} concurrent downloads, "
        "so a finished download never waits for a free process."
    )

# For comparison: scheduling="partitions" (round-robin, one partition at a time per account)
account_parts = [list(range(a, len(spatial_parts) + 1, account_count)) for a in range(1, account_count + 1)]
parts_per_account = pd.Series([len(parts) for parts in account_parts]).value_counts().sort_index(ascending=False)
initial_jobs = sum(min(OPENEO_JOBS_PER_USER, batches_per_partition[parts[0]]) for parts in account_parts if parts)
share_text = "; ".join(f"{accounts} account(s) get {parts} partition(s)" for parts, accounts in parts_per_account.items())
partition_verdict = (
    f"`batches` starts {min(slot_count, total_batches)} jobs instead, so it makes better use of the accounts."
    if initial_jobs < min(slot_count, total_batches) else "Both modes start with the same number of jobs."
)

display(Markdown(f"""
**Inputs:** {len(spatial_parts)} partitions from `{CLUSTER_FIELD}`, {len(gdf_grouped):,} parcels,
`TILE_SIZE_METRES = {TILE_SIZE_METRES:,}`, `TILE_BUFFER_METRES = {TILE_BUFFER_METRES:,}`,
{account_count} accounts × {OPENEO_JOBS_PER_USER} jobs = {slot_count} slots, `BATCH_WORKERS = {BATCH_WORKERS}`.

### 1. Partitions → batches (remote jobs)

| Partitions | Batches each | Batches |
| --- | ---: | ---: |
{chr(10).join(split_rows)}
| **Total** | | **{total_batches:,}** |

Buffered cubes measure {sides.min():.1f}–{sides.max():.1f} km per side and
{batch_plan['area_km2'].min():,.0f}–{batch_plan['area_km2'].max():,.0f} km² (median ≈ {batch_plan['area_km2'].median():,.0f} km²).
Parcels per batch range from {batch_plan['parcels'].min():,} to {largest['parcels']:,} (partition {largest['partition']}).
The cubes request about {requested_km2:,.0f} km² in total, {requested_km2 / extent_km2:.1f} × the
{(east - west) / 1_000:.0f} × {(north - south) / 1_000:.0f} km study extent (≈ {extent_km2:,.0f} km²);
values above 1 mean neighbouring cubes overlap.

### 2. Batches → accounts (`scheduling="batches"`)

| Round | Queue positions | Partitions | Slots used |
| --- | --- | --- | --- |
{chr(10).join(round_rows)}

In round 1, slots claim batches in roughly account order (User 1 ≈ batches 1–{OPENEO_JOBS_PER_USER},
User 2 ≈ {OPENEO_JOBS_PER_USER + 1}–{2 * OPENEO_JOBS_PER_USER}, …), but thread timing can change the exact pairing.

### 3. Local processing during the remote queue

{local_note}

### For comparison: `scheduling="partitions"`

Whole partitions would be assigned round-robin: User *a* gets partitions *a*, *a* + {account_count}, *a* + {2 * account_count}, …
({share_text}). Each account works through its own partitions one at a time, so the run would start with
about {initial_jobs} concurrent jobs. {partition_verdict}
"""))


**Inputs:** 69 partitions from `cluster_grid`, 23,072 parcels,
`TILE_SIZE_METRES = 5,400`, `TILE_BUFFER_METRES = 1,700`,
14 accounts × 2 jobs = 28 slots, `BATCH_WORKERS = 2`.

### 1. Partitions → batches (remote jobs)

| Partitions | Batches each | Batches |
| --- | ---: | ---: |
| 2 partitions | 1 | 2 |
| 68 | 2 | 2 |
| 42, 54, 69 | 4 | 12 |
| 28 | 5 | 5 |
| 38, 59, 64, 65, 67 | 6 | 30 |
| 27, 50, 53, 56, 60, 62, 63 | 7 | 49 |
| 30 | 8 | 8 |
| 37, 39 | 9 | 18 |
| 7, 17, 20, 29, 33, 48 | 11 | 66 |
| 10, 31, 49, 51, 52 | 12 | 60 |
| 1, 21, 36, 57 | 13 | 52 |
| 2, 11, 41, 44, 45, 46, 58 | 14 | 98 |
| 4, 5, 18, 26, 32, 47, 55 | 15 | 105 |
| 3, 6, 8, 9, 12, 13, 14, 15, 16, 19, 23, 25, ... (+3) | 16 | 240 |
| 22, 24, 43 | 17 | 51 |
| **Total** | | **798** |

Buffered cubes measure 3.5–5.4 km per side and
12–28 km² (median ≈ 23 km²).
Parcels per batch range from 1 to 200 (partition 22).
The cubes request about 17,738 km² in total, 4.7 × the
62 × 61 km study extent (≈ 3,793 km²);
values above 1 mean neighbouring cubes overlap.

### 2. Batches → accounts (`scheduling="batches"`)

| Round | Queue positions | Partitions | Slots used |
| --- | --- | --- | --- |
| 1 (immediate) | batches 1–28 | 1–3 | 28 (2 per account) |
| 2 (as slots free) | batches 29–56 | 3–4 | 28 (2 per account) |
| 3–28 (as slots free) | batches 57–784 | 4–65 | 28 (2 per account) each |
| 29 (tail) | batches 785–798 | 65–69 | 14; 14 idle |

In round 1, slots claim batches in roughly account order (User 1 ≈ batches 1–2,
User 2 ≈ 3–4, …), but thread timing can change the exact pairing.

### 3. Local processing during the remote queue

Up to 28 cubes can download concurrently but only 2 are processed locally at once, so after the first round local throughput, not openEO capacity, may set the pace.

### For comparison: `scheduling="partitions"`

Whole partitions would be assigned round-robin: User *a* gets partitions *a*, *a* + 14, *a* + 28, …
(13 account(s) get 5 partition(s); 1 account(s) get 4 partition(s)). Each account works through its own partitions one at a time, so the run would start with
about 28 concurrent jobs. Both modes start with the same number of jobs.
